
#### Better Code upon request
The main thing was to force Gemini to split the code into smaller functions and to use yield instead of returning a list of all chunks.

My prompt to Gemini:  
```
Code is not readable
Please split out a function called process_pdf_file
yield a chunk entry from it (so that it becomes a generator and not return a list of all chunks
```

This is what I got:

First, I got a verbal response before the code:
```
You're right—keeping the entire logic in a single loop gets messy fast. Using a generator (with yield) is a much more professional way to handle this. It keeps your memory footprint low because you're only holding one chunk at a time instead of a massive list of all 10 PDFs' worth of data.

Here is the refactored, readable version of your Medcare processing script.
```

And then the code:  

In [ ]:
import os
import json
import fitz  # PyMuPDF
from langchain.text_splitter import RecursiveCharacterTextSplitter

# --- CONFIGURATION ---
INPUT_FOLDER = "./medcare_docs"
OUTPUT_FILE = "medcare_knowledge_base.jsonl"

# Initialize the splitter once
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

def process_pdf_file(file_path):
    """
    Generator: Opens a single PDF, chunks it, and yields 
    one dictionary entry at a time.
    """
    filename = os.path.basename(file_path)
    doc = fitz.open(file_path)
    
    for page_num, page in enumerate(doc):
        page_text = page.get_text("text")
        chunks = text_splitter.split_text(page_text)
        
        for i, chunk_content in enumerate(chunks):
            # Yielding prevents building a giant list in memory
            yield {
                "chunk_id": f"{filename}_{page_num}_{i}",
                "text": chunk_content.strip(),
                "metadata": {
                    "source_file": filename,
                    "page_number": page_num + 1,
                    "org": "Medcare"
                }
            }
    doc.close()

def main():
    if not os.path.exists(INPUT_FOLDER):
        print(f"Directory {INPUT_FOLDER} not found.")
        return

    print("🚀 Starting Medcare PDF Extraction...")
    
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        for filename in os.listdir(INPUT_FOLDER):
            if filename.lower().endswith(".pdf"):
                file_path = os.path.join(INPUT_FOLDER, filename)
                print(f"  🧬 Processing: {filename}")

                # Use the generator to stream chunks into the file
                for chunk_entry in process_pdf_file(file_path):
                    f.write(json.dumps(chunk_entry) + '\n')

    print(f"✅ Finished! Knowledge base saved to {OUTPUT_FILE}")

if __name__ == "__main__":
    main()